In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
import scanpy as sc
import pandas as pd
import numpy as np
import torch
import dgl
import random
from scipy.sparse import csr_matrix
from sklearn.metrics import adjusted_rand_score as ari_score
from sklearn.metrics.cluster import normalized_mutual_info_score
import os

In [5]:
import so as so

In [6]:
seed = 42
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
dgl.random.seed(seed)

In [7]:
sc.set_figure_params(dpi=150, figsize=(2, 2), frameon=False)    # TODO 是否画边框

In [8]:
adata = sc.read_h5ad('/dataset/5_3d_mouse_kidney/mouse_kidney3d_arrayseq.h5ad')
adata

AnnData object with n_obs × n_vars = 92769 × 15144
    obs: 'Tissue_ID', 'Subregion', 'subregion', 'region'
    var: 'gene_ids', 'feature_types', 'mt'
    uns: 'Subregion_colors', 'hvg', 'leiden', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial', 'spatial_2d', 'spatial_3d'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [9]:
adata.obs

,Tissue_ID,Subregion,subregion,region
AAACCCAGCTCG-0-0-0-0-0-0-0,9,Cortex-PCT,Cortex-PCT,Cortex
AAACCCGACGAG-0-0-0-0-0-0-0,9,Cortex-PCT,Cortex-PCT,Cortex
AAACCCGAGGTG-0-0-0-0-0-0-0,9,Cortex-PCT,Cortex-PCT,Cortex
AAACCGAATCTC-0-0-0-0-0-0-0,9,Cortex-PCT,Cortex-PCT,Cortex
AAACCGCACTTA-0-0-0-0-0-0-0,9,Fat,Fat,Fat
...,...,...,...,...
TTTGGGCATCTC-1,2,Cortex-PCT,Cortex-PCT,Cortex
TTTGGTAGACGC-1,2,Medulla,Medulla,Medulla
TTTGGTCAGCAG-1,2,Medulla,Medulla,Medulla
TTTGGTCAGGAC-1,2,Cortex-PCT,Cortex-PCT,Cortex


In [10]:
adata.X.max()

4.628287

In [11]:
data_list = [adata[adata.obs.loc[:, 'Tissue_ID'] == batch, :] for batch in adata.obs.loc[:, 'Tissue_ID'].unique()]
data_list

[View of AnnData object with n_obs × n_vars = 12127 × 15144
     obs: 'Tissue_ID', 'Subregion', 'subregion', 'region'
     var: 'gene_ids', 'feature_types', 'mt'
     uns: 'Subregion_colors', 'hvg', 'leiden', 'log1p', 'neighbors', 'pca', 'umap'
     obsm: 'X_pca', 'X_umap', 'spatial', 'spatial_2d', 'spatial_3d'
     varm: 'PCs'
     obsp: 'connectivities', 'distances',
 View of AnnData object with n_obs × n_vars = 11247 × 15144
     obs: 'Tissue_ID', 'Subregion', 'subregion', 'region'
     var: 'gene_ids', 'feature_types', 'mt'
     uns: 'Subregion_colors', 'hvg', 'leiden', 'log1p', 'neighbors', 'pca', 'umap'
     obsm: 'X_pca', 'X_umap', 'spatial', 'spatial_2d', 'spatial_3d'
     varm: 'PCs'
     obsp: 'connectivities', 'distances',
 View of AnnData object with n_obs × n_vars = 12712 × 15144
     obs: 'Tissue_ID', 'Subregion', 'subregion', 'region'
     var: 'gene_ids', 'feature_types', 'mt'
     uns: 'Subregion_colors', 'hvg', 'leiden', 'log1p', 'neighbors', 'pca', 'umap'
     obsm: 

In [12]:
for data in data_list:
    sc.pp.filter_cells(data, min_genes=1)
    sc.pp.filter_genes(data, min_cells=1)
    print(data.X.max())

4.6051702
4.514896
4.628287
4.60137
4.5735183
4.60137
4.577862
4.5971284


In [13]:
batch_names = adata.obs.loc[:, 'Tissue_ID'].unique()
batch_names

['9', '10', '7', '8', '6', '5', '4', '2']
Categories (8, object): ['2', '4', '5', '6', '7', '8', '9', '10']

In [14]:
batch_names = ['2', '4', '5', '6', '7', '8', '9', '10']

In [15]:
batch_key = 'batch'
adata = so.pp.process_adata(data_list, label=batch_key, keys=batch_names, n_top_features=3000, target_sum=1e4, scale=False, norm=False)
adata

AnnData object with n_obs × n_vars = 92769 × 3000
    obs: 'Tissue_ID', 'Subregion', 'subregion', 'region', 'n_genes', 'original_index', 'spot_quality', 'batch'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'mu_bg', 'std_bg'
    obsm: 'X_pca', 'X_umap', 'spatial', 'spatial_2d', 'spatial_3d'
    layers: 'counts'

In [16]:
rad_cutoff_list = [0.05] * len(data_list)
adata, g = so.pp.process_graph(adata, data_list, cutoff_list=rad_cutoff_list)

cal spatial net in data_list
------Calculating spatial graph...
The graph contains 70504 edges, 12127 cells.
5.8138 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 65708 edges, 11247 cells.
5.8423 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 73652 edges, 12712 cells.
5.7939 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 71524 edges, 12291 cells.
5.8192 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 67364 edges, 11685 cells.
5.7650 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 65168 edges, 11225 cells.
5.8056 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 65082 edges, 11273 cells.
5.7733 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 59442 edges, 10209 cells.
5.8225 neighbors per cell on average.


In [17]:
path_results = f'./log/integration_radius/'

In [18]:
adata_int = so.model.SpaVCCA(adata=adata,
                            n_epoch=5,
                            batch_size=1024,
                            verbose=False,
                            gpu=0,
                            random_seed=42,
                            output_dir=path_results,
                            num_heads=2,
                            h_dim=16,
                            reconstruct_loss='orig',
                            alpha_mmd=1,
                            )

clip
reconstruct_loss == orig


Epochs: 100%|█| 5/5 [00:30<00:00,  6.11s/it, recon_loss=25.6830, kl_loss=1.1256, mmd2=0.0021, graph_loss=0.0000, lr: 0.0
Infer latent: 100%|█████████████████████████████████████████████████████████████████████| 91/91 [00:02<00:00, 35.54it/s]


In [19]:
del adata_int.uns['adj']

In [20]:
adata_int.write_h5ad(f'{path_results}adata_int.h5ad', compression='gzip')